In [43]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from meteo_api import MeteoAPI
from datetime import datetime
from pathlib import Path

from constants import Col
from utils import prepare_df

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
base_url = "https://api.meteo.lt/v1"
station_code = "kauno-ams"
city_code = "kaunas"

start_date = datetime(2025, 9, 12)
end_date = datetime(2026, 9, 12)

api = MeteoAPI(base_url, city_code, station_code)

In [45]:
load = True
year_path = Path("data/year.csv")

if year_path.exists() and load:
    obs_df = pd.read_csv(year_path)
    obs_df = prepare_df(obs_df)
else:
    obs_df = api.get_observations(start_date, end_date)
    obs_df.to_csv(year_path)

obs_df[Col.RELATIVE_HUMIDITY] = obs_df[Col.RELATIVE_HUMIDITY].astype('Int64')

In [46]:
obs_df.head()

,airTemperature,feelsLikeTemperature,windSpeed,windGust,windDirection,cloudCover,seaLevelPressure,relativeHumidity,precipitation,snowDepth,conditionCode,timeLocal
timeUtc,,,,,,,,,,,,
2025-09-12 00:00:00+00:00,15.7,15.7,7.2,11.4,170.0,100.0,1014.7,95,2.2,0,light-rain,2025-09-12 03:00:00+03:00
2025-09-12 01:00:00+00:00,15.7,15.7,6.0,9.9,171.0,100.0,1015.0,95,0.6,0,rain,2025-09-12 04:00:00+03:00
2025-09-12 02:00:00+00:00,15.6,15.6,5.4,8.7,171.0,100.0,1015.2,95,0.0,0,cloudy,2025-09-12 05:00:00+03:00
2025-09-12 03:00:00+00:00,15.6,15.6,5.1,7.8,170.0,100.0,1015.4,95,0.1,0,light-rain,2025-09-12 06:00:00+03:00
2025-09-12 04:00:00+00:00,15.6,15.6,4.7,7.3,167.0,100.0,1015.6,95,0.5,0,light-rain,2025-09-12 07:00:00+03:00


In [47]:
obs_df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 8767 entries, 2025-09-12 00:00:00+00:00 to 2026-09-12 06:00:00+00:00
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype                         
---  ------                --------------  -----                         
 0   airTemperature        8767 non-null   float64                       
 1   feelsLikeTemperature  8257 non-null   float64                       
 2   windSpeed             8738 non-null   float64                       
 3   windGust              8730 non-null   float64                       
 4   windDirection         8738 non-null   float64                       
 5   cloudCover            8636 non-null   float64                       
 6   seaLevelPressure      8767 non-null   float64                       
 7   relativeHumidity      8286 non-null   Int64                         
 8   precipitation         8767 non-null   float64                       
 9   snowDepth             8767 non-null  

In [52]:
obs_df.isna().sum()

airTemperature            0
feelsLikeTemperature    510
windSpeed                29
windGust                 37
windDirection            29
cloudCover              131
seaLevelPressure          0
relativeHumidity        481
precipitation             0
snowDepth                 0
conditionCode             0
timeLocal                 0
dtype: int64

In [48]:
obs_df.describe()

,airTemperature,feelsLikeTemperature,windSpeed,windGust,windDirection,cloudCover,seaLevelPressure,relativeHumidity,precipitation,snowDepth
count,8767.000000,8257.000000,8738.000000,8730.000000,8738.000000,8636.000000,8767.000000,8286.0,8767.000000,8767.000000
mean,7.385001,5.613177,3.234390,6.096186,203.334401,62.352825,1015.459644,78.463191,0.071587,3.463671
std,10.002713,12.099309,1.664398,2.984682,99.342625,42.810689,8.917664,16.525455,0.507155,7.629498
min,-27.100000,-32.600000,0.100000,0.600000,0.000000,0.000000,984.300000,20.0,0.000000,0.000000
25%,1.150000,-2.300000,2.000000,3.800000,146.000000,0.000000,1010.600000,69.0,0.000000,0.000000
50%,8.100000,6.500000,3.000000,5.700000,210.000000,88.000000,1016.200000,83.0,0.000000,0.000000
75%,14.700000,15.000000,4.300000,8.000000,286.000000,100.000000,1020.700000,92.0,0.000000,0.000000
max,32.500000,33.500000,13.100000,25.900000,360.000000,100.000000,1043.500000,99.0,24.900000,29.000000


In [ ]:
day_mask = obs_df[Col.TIME_LOCAL].dt.hour.between(8, 20)
day_df = obs_df[day_mask]
night_df = obs_df[~day_mask]

In [79]:
weekend_mask = obs_df[Col.TIME_LOCAL].dt.dayofweek.between(5, 6)
weekend_df = obs_df[weekend_mask]

weekend_df["week"] = weekend_df[Col.TIME_LOCAL].dt.isocalendar().week
weekend_df["year"] = weekend_df[Col.TIME_LOCAL].dt.isocalendar().year

weekend_precip = weekend_df.groupby(["week", "year"])[Col.PRECIPITATION].sum()

rainy_weekends = sum(weekend_precip > 0)
print(rainy_weekends)

32


In [80]:
print(f"Vidutinė metų temperatūra: {obs_df[Col.AIR_TEMPERATURE].mean():.2f} C")
print(f"Vidutinė metų dienos temperatūra: {day_df[Col.AIR_TEMPERATURE].mean():.2f} C")
print(f"Vidutinė metų nakties temperatūra: {night_df[Col.AIR_TEMPERATURE].mean():.2f} C")
print(f"Lietingi savaitgaliai per metus: {rainy_weekends}")

Vidutinė metų temperatūra: 7.39 C
Vidutinė metų dienos temperatūra: 8.85 C
Vidutinė metų nakties temperatūra: 5.66 C
Lietingi savaitgaliai per metus: 32
